In [ ]:
!pip install pandas numpy matplotlib scikit-learn transformers torch prophet
import os
print("Libraries installed and ready!")

# ***Cell 1 Model Comparison Table***

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

comparison_data = [
    {"Model": "Ridge Regression", "Params": 154, "Train Time (s)": 0.05, "Inf Time (ms)": 0.1,
     "MAE": 0.0421, "RMSE": 0.0512, "MAPE (%)": 4.5, "R2": 0.65},
    {"Model": "Random Forest", "Params": "N/A", "Train Time (s)": 2.10, "Inf Time (ms)": 5.2,
     "MAE": 0.0315, "RMSE": 0.0420, "MAPE (%)": 3.8, "R2": 0.78},
    {"Model": "LSTM", "Params": 43105, "Train Time (s)": 15.4, "Inf Time (ms)": 1.2,
     "MAE": 0.0284, "RMSE": 0.0351, "MAPE (%)": 3.1, "R2": 0.85},
    {"Model": "GRU", "Params": 32801, "Train Time (s)": 12.8, "Inf Time (ms)": 1.0,
     "MAE": 0.0279, "RMSE": 0.0345, "MAPE (%)": 3.0, "R2": 0.86},
    {"Model": "TCN v2 (dilated residual)*", "Params": 220321, "Train Time (s)": 29.5, "Inf Time (ms)": 1.9,
     "MAE": 0.0295, "RMSE": 0.0368, "MAPE (%)": 3.3, "R2": 0.83},
]

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

print("* TCN v2 replaces the earlier 2-layer TCN (38,945 params). Params are exact (verified from the")
print("  model definition). Train/Inference time are the v1 placeholder figures scaled by the real speed")
print("  ratio measured when both versions were run back-to-back on identical data (3.47x train, 3.15x")
print("  inference) -- confirm on your own hardware. MAE/RMSE/MAPE/R2 for TCN v2 are still the v1 figures")
print("  and need to be re-measured on the real dataset once training completes.")

# ***Cell 2 Model Comparison & Discussion***

In [ ]:
discussion_report = """
**Complexity & Runtime** --- Linear baselines are fast but underfit. LSTM/GRU stay the fast, cheap sequence baselines. TCN v2 (dilated residual, 220K params) is now the *largest and slowest* model to train (~3.5x slower than the old TCN) --- it has traded away its old "fastest deep model" advantage for a much wider receptive field (dilation 1-2-4) and far more capacity.

**Overfitting** --- Limited battery cells already caused a train/val gap in sequence models; TCN v2 has ~4.8x more parameters than v1 on the *same* small dataset, so overfitting risk is higher, not lower. Watch the train/val loss gap closely -- dropout (0.2) is already in place, raise it if the gap widens.

**Data Limitations** --- Few unique battery cells and no per-cell chemistry metadata means all models share the same ceiling: they can only generalize as well as the cell-to-cell variation allows. More capacity does not fix this.

**Failure Patterns** --- All models still struggle at (1) capacity-regeneration spikes and (2) the "knee" transition, smoothing out the true severity of the drop. TCN v2's longer receptive field (reaches 7 steps back at dilation 4) is the most direct attempt yet at fixing the knee-lag problem, since it can see further into each cell's recent history than v1 could.

**Shortlist** --- **GRU**: still the best accuracy-per-parameter, cheapest to train, safe default. **TCN v2**: highest capacity and widest receptive field, a real candidate for the knee-transition problem specifically, but must be checked for overfitting given the parameter jump before it earns a place over GRU.

**Decision** --- Proceed with GRU (efficiency baseline) and TCN v2 (capacity/receptive-field candidate) for deeper experiments. Re-run TCN v2 on the real dataset first to confirm its MAE/RMSE actually improve and that the train/val gap stays controlled -- do not assume the accuracy row above until that is confirmed.
"""
display(Markdown(discussion_report))